In [14]:
import requests
from bs4 import BeautifulSoup

Response = requests.get("https://www.honda-indonesia.com/dealers/")
print(Response.status_code)
# print(Response.text)

200


In [15]:
soup = BeautifulSoup(Response.text, 'html.parser')
table_blocks = soup.find_all('div', class_='tw-space-y-4')
print("Number of countries found: ", len(table_blocks))

Number of countries found:  194


In [16]:
import requests
from bs4 import BeautifulSoup
import re
import pandas as pd

# --- Request ---
url = "https://www.honda-indonesia.com/dealers/"
headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"}
response = requests.get(url, headers=headers, timeout=15)
response.raise_for_status()
print(f"Status: {response.status_code}")

soup = BeautifulSoup(response.text, 'html.parser')

# --- Cari semua konten provinsi (div dengan x-show="open === X") ---
province_content_divs = soup.find_all('div', attrs={'x-show': lambda x: x and 'open ===' in x})

print(f"Jumlah provinsi terdeteksi: {len(province_content_divs)}")

result = []

for content_div in province_content_divs:
    try:
        # 1. Ambil ID provinsi dari x-show
        x_show = content_div.get('x-show')
        province_id = re.search(r'open === (\d+)', x_show).group(1)

        # 2. Cari button provinsi (sebelum content_div)
        button = content_div.find_previous('button')
        if not button:
            continue

        span = button.find('span', class_='tw-font-bold')
        province_name = span.get_text(strip=True) if span else "Unknown"

        # 3. Cari semua blok dealer di dalam content_div
        dealer_blocks = content_div.find_all('div', class_='tw-space-y-4')

        for block in dealer_blocks:
            try:
                # Nama Dealer
                name_tag = block.find('a', href=lambda x: x and '/dealers/' in x)
                dealer = name_tag.get_text(strip=True) if name_tag else "N/A"

                # Alamat
                alamat_tag = block.find('div', class_='tw-text-gray-500')
                alamat = alamat_tag.get_text(strip=True) if alamat_tag else "Tidak tersedia"

                # Koordinat
                direction_tag = None
                for a in block.find_all('a'):
                    if a.get_text(strip=True) == "Direction":
                        direction_tag = a
                        break

                lat, long = "N/A", "N/A"
                if direction_tag and direction_tag.get('href'):
                    href = direction_tag['href']
                    match = re.search(r'q=([-\d.]+),([-\d.]+)', href)
                    if match:
                        lat, long = match.group(1), match.group(2)

                # Simpan
                result.append({
                    'Provinsi': province_name,
                    'Dealer': dealer,
                    'Alamat': alamat,
                    'Latitude': lat,
                    'Longitude': long
                })

            except Exception as e:
                print(f"Error parsing dealer block: {e}")
                continue

    except Exception as e:
        print(f"Error parsing province: {e}")
        continue

# --- Output ---
print("\n" + "="*100)
print("CONTOH 5 DEALER PERTAMA".center(100))
print("="*100)
for item in result[:5]:
    print(f"Provinsi   : {item['Provinsi']}")
    print(f"Dealer     : {item['Dealer']}")
    print(f"Alamat     : {item['Alamat']}")
    print(f"Lat/Long   : {item['Latitude']}, {item['Longitude']}")
    print("-" * 100)


Status: 200
Jumlah provinsi terdeteksi: 32

                                      CONTOH 5 DEALER PERTAMA                                       
Provinsi   : Bali
Dealer     : Honda Bintang Tabanan
Alamat     : 
Lat/Long   : N/A, N/A
----------------------------------------------------------------------------------------------------
Provinsi   : Bali
Dealer     : Honda Cokroaminoto
Alamat     : Jl. Cokroaminoto No. 168
Lat/Long   : N/A, N/A
----------------------------------------------------------------------------------------------------
Provinsi   : Bali
Dealer     : Honda Denpasar Agung
Alamat     : Jl. Hayam Wuruk No. 40
Lat/Long   : -8.65731, 115.22685
----------------------------------------------------------------------------------------------------
Provinsi   : Bali
Dealer     : Honda Dewata Motor
Alamat     : Jl. Imam Bonjol No. 104
Lat/Long   : -8.66767, 115.20567
----------------------------------------------------------------------------------------------------
Provinsi   

In [17]:
# --- Simpan ke Excel ---
import pandas as pd

df = pd.DataFrame(result)

# Add headers
df.columns = ['Provinsi', 'Dealer', 'Alamat', 'Latitude', 'Longitude']

# Save to 
df

,Provinsi,Dealer,Alamat,Latitude,Longitude
0,Bali,Honda Bintang Tabanan,,N/A,N/A
1,Bali,Honda Cokroaminoto,Jl. Cokroaminoto No. 168,N/A,N/A
2,Bali,Honda Denpasar Agung,Jl. Hayam Wuruk No. 40,-8.65731,115.22685
3,Bali,Honda Dewata Motor,Jl. Imam Bonjol No. 104,-8.66767,115.20567
4,Banten,Honda Arta Cikupa,"Jl. Raya Serang KM 14, Cikupa",-6.22295,106.529
...,...,...,...,...,...
187,Sumatera Utara,Honda Arista Siantar,,N/A,N/A
188,Sumatera Utara,Honda Arista SM Raja,Jl. Sisingamangaraja Km. 5.5 No. 2,3.54615,98.69831
189,Sumatera Utara,Honda IDK 1,Jl.Glugur By Pass No. 85,3.60232,98.66879
190,Sumatera Utara,Honda IDK 2,Jl. Sei Batang Hari No. 22-24,3.58511,98.65167


In [ ]:
# --- Scraping Koordinat dengan Selenium untuk Dealer yang Belum Terisi ---
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import re
import time

# Pastikan DataFrame df sudah ada
if 'df' not in locals() and 'df' not in globals():
    if 'result' in locals() or 'result' in globals():
        df = pd.DataFrame(result)
        df.columns = ['Provinsi', 'Dealer', 'Alamat', 'Latitude', 'Longitude']
    else:
        raise ValueError("DataFrame 'df' tidak ditemukan. Jalankan Cell 2 dan Cell 3 terlebih dahulu.")

# Filter dealer yang belum punya koordinat
mask = (df['Latitude'] == 'N/A') | (df['Latitude'].isna()) | (df['Longitude'] == 'N/A') | (df['Longitude'].isna())
dealers_to_scrape = df[mask].copy()

print(f"Total dealer: {len(df)}")
print(f"Perlu di-scrape: {len(dealers_to_scrape)} | Sudah ada koordinat: {len(df[~mask])}")
if len(dealers_to_scrape) > 0:
    print(f"\nContoh dealer yang akan di-scrape:")
    print(dealers_to_scrape[['Dealer', 'Alamat', 'Latitude', 'Longitude']].head())

Total dealer: 192
Perlu di-scrape: 50 | Sudah ada koordinat: 142

Contoh dealer yang akan di-scrape:
                   Dealer                                             Alamat  \
0   Honda Bintang Tabanan                                                      
1      Honda Cokroaminoto                           Jl. Cokroaminoto No. 168   
5      Honda Auto Cilegon                                                      
17  Honda Anugerah Bantul  Jl. Ringroad Barat, Tamantirto, Kasihan, Kabup...   
29     Honda Maju Pd Gede                                                      

   Latitude Longitude  
0       N/A       N/A  
1       N/A       N/A  
5       N/A       N/A  
17      N/A       N/A  
29      N/A       N/A  


In [19]:
# --- Setup Selenium WebDriver ---
from selenium.webdriver.chrome.options import Options

chrome_options = Options()
chrome_options.add_argument('--start-maximized')
chrome_options.add_argument('--disable-blink-features=AutomationControlled')
chrome_options.add_experimental_option("excludeSwitches", ["enable-automation"])
chrome_options.add_experimental_option('useAutomationExtension', False)

driver = webdriver.Chrome(options=chrome_options)
print("WebDriver Chrome sudah diinisialisasi")


WebDriver Chrome sudah diinisialisasi


In [20]:
# --- Fungsi untuk extract koordinat dari URL Google Maps ---
def extract_coordinates_from_url(url):
    """Extract koordinat dari URL Google Maps dengan berbagai format"""
    try:
        patterns = [
            r'@(-?\d+\.?\d*),(-?\d+\.?\d*)',      # Format @lat,long
            r'[?&]q=(-?\d+\.?\d*),(-?\d+\.?\d*)', # Format q=lat,long
            r'/dir/(-?\d+\.?\d*),(-?\d+\.?\d*)',   # Format /dir/lat,long
            r'center=(-?\d+\.?\d*),(-?\d+\.?\d*)'  # Format center=lat,long
        ]
        
        for pattern in patterns:
            match = re.search(pattern, url)
            if match:
                lat, lon = match.group(1), match.group(2)
                if -90 <= float(lat) <= 90 and -180 <= float(lon) <= 180:
                    return lat, lon
    except:
        pass
    return None, None


In [21]:
# --- Fungsi untuk scraping koordinat dari Google Maps ---
def scrape_coordinates(dealer_name, alamat=""):
    """Search dealer di Google Maps dan extract koordinat dari URL"""
    try:
        # Buat search query
        query = f"{dealer_name} {alamat} Indonesia" if alamat and alamat.strip() and alamat != "Tidak tersedia" else f"{dealer_name} Indonesia"
        maps_url = f"https://www.google.com/maps/search/{query.replace(' ', '+')}"
        
        driver.get(maps_url)
        time.sleep(4)
        
        # Extract koordinat dari URL
        current_url = driver.current_url
        lat, lon = extract_coordinates_from_url(current_url)
        
        if lat and lon:
            return lat, lon
        
        # Fallback: klik hasil pertama jika belum dapat koordinat
        try:
            first_result = WebDriverWait(driver, 5).until(
                EC.element_to_be_clickable((By.CSS_SELECTOR, "div[role='article']:first-of-type"))
            )
            first_result.click()
            time.sleep(3)
            current_url = driver.current_url
            lat, lon = extract_coordinates_from_url(current_url)
            if lat and lon:
                return lat, lon
        except:
            pass
        
        return None, None
    except Exception as e:
        print(f"Error scraping {dealer_name}: {str(e)}")
        return None, None


In [22]:
# --- Loop untuk scraping semua dealer yang belum punya koordinat ---
if len(dealers_to_scrape) > 0:
    success_count = failed_count = 0
    total = len(dealers_to_scrape)
    
    print(f"\n{'='*80}")
    print(f"MULAI SCRAPING {total} DEALER")
    print(f"{'='*80}\n")
    
    for i, (idx, row) in enumerate(dealers_to_scrape.iterrows(), 1):
        dealer_name = row['Dealer']
        alamat = row['Alamat'] if pd.notna(row['Alamat']) else ""
        
        print(f"[{i}/{total}] {dealer_name}")
        lat, lon = scrape_coordinates(dealer_name, alamat)
        
        if lat and lon:
            df.at[idx, 'Latitude'] = lat
            df.at[idx, 'Longitude'] = lon
            success_count += 1
            print(f"  ✓ Lat={lat}, Long={lon}")
        else:
            failed_count += 1
            print(f"  ✗ Gagal")
        
        time.sleep(2)
        
        if i % 10 == 0:
            df.to_excel('honda_dealers_indonesia.xlsx', index=False)
            print(f"\n💾 Progress: {success_count} sukses, {failed_count} gagal\n")
    
    print(f"\n{'='*80}")
    print(f"SCRAPING SELESAI!")
    print(f"Sukses: {success_count} | Gagal: {failed_count} | Rate: {(success_count/total*100):.1f}%")
    print(f"{'='*80}")
    
    df.to_excel('honda_dealers_indonesia.xlsx', index=False)
    mask_final = (df['Latitude'] == 'N/A') | (df['Latitude'].isna()) | (df['Longitude'] == 'N/A') | (df['Longitude'].isna())
    print(f"\n📊 Dealer dengan koordinat: {len(df[~mask_final])} | Tanpa koordinat: {len(df[mask_final])}")
else:
    print("Tidak ada dealer yang perlu di-scrape")



MULAI SCRAPING 50 DEALER

[1/50] Honda Bintang Tabanan
  ✓ Lat=-8.5530323, Long=115.1362066
[2/50] Honda Cokroaminoto
  ✓ Lat=-8.6412412, Long=115.2092679
[3/50] Honda Auto Cilegon
  ✓ Lat=-6.0487, Long=106.0615265
[4/50] Honda Anugerah Bantul
  ✓ Lat=-7.8090155, Long=110.3244686
[5/50] Honda Maju Pd Gede
  ✓ Lat=-6.284764, Long=106.906087
[6/50] Honda Thamrin Jambi
  ✓ Lat=-1.6210389, Long=103.6305954
[7/50] Honda IBRM Subang
  ✗ Gagal
[8/50] Honda Kumala Cikampek
  ✗ Gagal
[9/50] Honda LPPM Kuningan
  ✗ Gagal
[10/50] Honda Perdana Soreang
  ✗ Gagal

💾 Progress: 6 sukses, 4 gagal

[11/50] Honda Manunggal Brebes
  ✗ Gagal
[12/50] Honda Mandalasena Blitar
  ✗ Gagal
[13/50] Honda Prisma HR Muhammad
  ✗ Gagal
[14/50] Honda Trio Pangkalan Bun
  ✗ Gagal
[15/50] Honda Trio Sampit
  ✓ Lat=-2.5481482, Long=112.9528615
[16/50] Honda Amartha Berau
  ✓ Lat=2.1519295, Long=117.4958041
[17/50] Honda Amartha Bontang
  ✓ Lat=0.1219149, Long=117.483988
[18/50] Honda Amartha Sangatta
  ✓ Lat=0.5091975

In [23]:
# --- Tutup WebDriver ---
driver.quit()
print("WebDriver telah ditutup")


WebDriver telah ditutup
